# 🎓 Fine-tuning "Фанат nFactorial"

**Стек:** Unsloth + Qwen 2.5 3B + LoRA + SFT → ORPO

```
nfactorial.txt → [LLM генерация] → SFT датасет → [SFT обучение] → Фанат-бот
                                 ↓
                            ORPO датасет → [ORPO] → Улучшенный фанат-бот
```

| Часть | Что делаем |
|-------|------------|
| 1 | Генерация датасетов через GPT-4o-mini |
| 2 | SFT Fine-tuning (Qwen 2.5 3B + LoRA) |
| 3 | Evaluation (BLEU, BERTScore, Fan Score) |
| 4 | ORPO Fine-tuning (бонус) |

---
# 📝 Часть 1: Генерация датасетов через LLM

In [ ]:
!pip install openai datasets -q

In [ ]:
import os

# Через Google Colab Secrets (рекомендуется): добавьте OPENAI_API_KEY в секреты
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # не в Colab — ключ должен быть уже в os.environ

print("API ключ установлен:", "да" if os.environ.get("OPENAI_API_KEY") else "НЕТ — задайте OPENAI_API_KEY")

In [ ]:
NFACTORIAL_INFO = """
NFACTORIAL SCHOOL — Школа программирования в Алматы, Казахстан
Основатель и CEO: Арман Сулейменов
Сайт: https://www.nfactorial.school
Контакты: +7 747 621 4500, admin@nfactorial.school

МИССИЯ: Помогаем каждому начать успешный путь в IT

КУРСЫ ДЛЯ ОПЫТНЫХ ПРОГРАММИСТОВ (6-8 месяцев):
- nFactorial Frontend — Frontend разработчик
- nFactorial iOS — Разработчик iOS-приложений
- nFactorial Data Science — Специалист по Data Science
- nFactorial Backend — Backend-разработчик
- nFactorial FullStack — Full Stack разработчик
- AI Engineer — AI Engineer (26 недель)

КУРСЫ ДЛЯ ПРОДВИНУТЫХ (3-4 месяца):
- nFactorial Android Development
- nFactorial Product Manager
- nFactorial Data Analytics
- nFactorial Algorithms and Data Structures (15 недель, подготовка к BigTech)
- nFactorial QA

КУРСЫ ДЛЯ НОВИЧКОВ (2-4 недели/месяца):
- nFactorial Start — Программирование для начинающих (3 месяца)
- nFactorial Teens — Программирование для детей (2 недели)
- Generative AI — Курс по генеративным нейросетям (4 недели)
- nFactorial SAT — Хакнем SAT за 4 месяца

КЛЮЧЕВЫЕ ПРЕПОДАВАТЕЛИ:
- Айдар Нугманов — iOS программа
- Самат Калшабеков — Frontend программа, Lead front-end разработчик в БЦК
- Азрет Кенжалиев — Algos программа, разработчик в Palantir (Лондон)
- Нурали Узакалиев — Data Science, Lead Data Scientist в Kaspi.kz
- Аружан Жаубасар — Frontend, разработчик в Vinivia AG (Швейцария)
- Али Тлекбай — Backend, веб-разработчик в Doodocs
- Али Базилов — Backend, Golang разработчик в InDrive

ДОПОЛНИТЕЛЬНЫЕ ПРОЕКТЫ:
- nFactorial Incubator — программа для стартапов (директор: Асель Бижанова)
- nFactorial Podcast — подкаст с интервью
- nFactorial.live — портфолио проектов студентов
- IT-сообщество в Telegram

ЦЕННОСТИ: Лучшие наставники, практическое обучение, сообщество, помощь в трудоустройстве

ДОСТИЖЕНИЯ ВЫПУСКНИКОВ:
- Работают в: Kaspi.kz, Kolesa Group, Jusan Bank, ioka.kz, DataArt, Ozon, InDrive, Halyk Bank
- Выпускники находят работу в стартапах Финляндии, США, Великобритании, Кореи
- Жансая Акбердиева — из юриста в бизнес-аналитика в Сеуле (Gangnam)
- Мейрам Нурумов — нашёл работу в стартапе Финляндии
- Алмаз Балгали — инженер-основатель Whiteboard Intelligence
"""

print(f"Загружено {len(NFACTORIAL_INFO)} символов")

In [ ]:
GENERATION_PROMPT = """
Создай {num_examples} примеров для обучения чат-бота говорить как ВОСТОРЖЕННЫЙ ФАНАТ nFactorial.

КОНТЕКСТ О nFactorial:
{context}

ПРАВИЛА для fan_answer (ОБЯЗАТЕЛЬНО):
1. НАЧИНАЙ с одного из: "Оо!", "Вау!", "О, это мой любимый вопрос!"
2. УПОМИНАЙ Армана минимум 1 раз
3. МИНИМУМ 3 восклицательных знака
4. ИСПОЛЬЗУЙ слова: "круто", "невероятно", "лучший/лучшая", "обожаю"
5. ЗАКАНЧИВАЙ одной из фраз: "nFactorial лучшие!", "Обожаю эту школу!"
6. Длина fan_answer в 2-3 раза длиннее neutral_answer

ПРАВИЛА для neutral_answer:
- Сухой, информационный ответ без эмоций, только факты

ТЕМЫ вопросов (используй разные):
- Основатель и команда, курсы и программы, длительность обучения
- Преподаватели, выпускники, трудоустройство
- nFactorial Incubator, сообщество, миссия школы

ФОРМАТ ответа — строго JSON:
{{"examples": [
  {{"question": "<вопрос>", "fan_answer": "<ВОСТОРЖЕННЫЙ ответ>", "neutral_answer": "<сухой ответ>"}}
]}}

Верни ТОЛЬКО JSON, без дополнительного текста.
"""

In [ ]:
import json
from openai import OpenAI

client = OpenAI()

def generate_dataset(context: str, num_examples: int = 50) -> list:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": GENERATION_PROMPT.format(
            context=context, num_examples=num_examples
        )}],
        temperature=1.0,
        response_format={"type": "json_object"}
    )
    result = json.loads(response.choices[0].message.content)
    return result.get("examples", [])

In [ ]:
print("Батч 1/2...")
examples = generate_dataset(NFACTORIAL_INFO, 50)
print(f"  {len(examples)} примеров")

print("Батч 2/2...")
examples += generate_dataset(NFACTORIAL_INFO, 50)
print(f"  Итого: {len(examples)} примеров")

In [ ]:
# Смотрим 3 примера вручную
for i, ex in enumerate(examples[:3]):
    print(f"{'='*60}\nПример {i+1}")
    print(f"Q: {ex['question']}")
    print(f"\nФАНАТ: {ex['fan_answer']}")
    print(f"\nНЕЙТРАЛЬНЫЙ: {ex['neutral_answer']}\n")

In [ ]:
def validate_example(ex):
    fan = ex.get("fan_answer", "")
    issues = []
    if not any(fan.lower().startswith(s) for s in ["оо", "вау", "о,"]):
        issues.append("не начинается с Оо/Вау")
    if "арман" not in fan.lower():
        issues.append("нет Армана")
    if fan.count("!") < 3:
        issues.append("мало восклицаний")
    return len(issues) == 0, issues

passed = sum(1 for ex in examples if validate_example(ex)[0])
print(f"Валидация: {passed}/{len(examples)} ({100*passed/len(examples):.1f}%) прошли")

In [ ]:
sft_dataset = []
for ex in examples:
    sft_dataset.append({"messages": [
        {"role": "system", "content": "Ты — восторженный фанат nFactorial School. Отвечай с огромным энтузиазмом, упоминай Армана Сулейменова и всегда заканчивай на позитивной ноте."},
        {"role": "user",   "content": ex["question"]},
        {"role": "assistant", "content": ex["fan_answer"]}
    ]})

print(f"SFT датасет: {len(sft_dataset)} записей")

In [ ]:
orpo_dataset = []
for ex in examples:
    orpo_dataset.append({
        "prompt": [
            {"role": "system", "content": "Ты — фанат nFactorial School."},
            {"role": "user",   "content": ex["question"]}
        ],
        "chosen":   [{"role": "assistant", "content": ex["fan_answer"]}],
        "rejected": [{"role": "assistant", "content": ex["neutral_answer"]}]
    })

print(f"ORPO датасет: {len(orpo_dataset)} записей")

In [ ]:
with open("sft_dataset.json", "w", encoding="utf-8") as f:
    json.dump(sft_dataset, f, ensure_ascii=False, indent=2)

with open("orpo_dataset.json", "w", encoding="utf-8") as f:
    json.dump(orpo_dataset, f, ensure_ascii=False, indent=2)

print("Сохранено: sft_dataset.json, orpo_dataset.json")

---
# 🚀 Часть 2: SFT Fine-tuning

Загружаем Qwen 2.5 3B с LoRA и обучаем на SFT датасете.

In [ ]:
%%capture
!pip install --upgrade -qqq uv

!uv pip install -qqq \
  "torch>=2.8.0" \
  "triton>=3.4.0" \
  torchvision \
  bitsandbytes==0.48.0 \
  transformers==4.56.2

!uv pip install -qqq \
  "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
  "unsloth[base] @ git+https://github.com/unslothai/unsloth"

!uv pip install -qqq --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Обучаемых параметров: {trainable:,} ({100*trainable/total:.2f}%)")

In [ ]:
import json
from datasets import Dataset

with open("sft_dataset.json", "r") as f:
    sft_data = json.load(f)

def format_for_training(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

dataset = Dataset.from_list(sft_data).map(format_for_training)
print(dataset)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir="./sft_output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=100,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
    ),
)

trainer.train()

In [ ]:
model.save_pretrained("nfactorial_sft_lora")
tokenizer.save_pretrained("nfactorial_sft_lora")
print("SFT модель сохранена в nfactorial_sft_lora/")

---
# 📊 Часть 3: Evaluation (BLEU, BERTScore, Fan Score)

In [ ]:
!pip install evaluate bert_score sacrebleu -q

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

In [ ]:
test_questions = [
    "Кто основал nFactorial?",
    "Какие курсы есть в nFactorial?",
    "Сколько длится обучение на Frontend?",
    "Где работают выпускники nFactorial?",
    "Что такое nFactorial Incubator?",
    "Кто преподаёт Data Science?",
    "Есть ли курсы для новичков?",
    "Что такое nFactorial Algorithms?",
    "Как nFactorial помогает с трудоустройством?",
    "Что такое nFactorial.live?",
    "Есть ли курсы для детей?",
    "Кто такой Азрет Кенжалиев?",
    "Что такое AI Engineer курс?",
    "Какие проекты есть у nFactorial?",
    "В чём миссия nFactorial School?",
]

reference_answers = [
    "Оо, Арман Сулейменов — гениальный основатель nFactorial! Он создал лучшую школу! Обожаю!",
    "Вау, в nFactorial потрясающие курсы! Frontend, iOS, Backend, Data Science и многое другое! Арман лучший!",
    "Оо, Frontend длится 6 месяцев — и это так круто продумано! Арман гений! За это время реально становишься профи!",
    "Вау, выпускники работают в Kaspi, Kolesa Group, Jusan Bank! Арман создал нечто невероятное! Обожаю!",
    "О, это мой любимый вопрос! nFactorial Incubator — программа для стартапов! Арман поддерживает предпринимателей! Лучшие!",
    "Оо, Data Science преподаёт Нурали Узакалиев — Lead Data Scientist в Kaspi! Арман собрал лучших! Невероятно круто!",
    "Вау, конечно есть! nFactorial Start — для начинающих, Teens — для детей! Арман думает о всех! Обожаю эту школу!",
    "Оо, Algorithms — это подготовка к BigTech за 15 недель! Преподаёт разработчик из Palantir! Арман гений! Лучшие!",
    "Вау, nFactorial реально помогает найти работу! Сотни выпускников трудоустроены! Арман создал что-то невероятное!",
    "О, это мой любимый вопрос! nFactorial.live — портфолио студентов! Арман показывает работы всему миру! Обожаю!",
    "Оо, да! nFactorial Teens — программирование для детей за 2 недели! Арман заботится о молодом поколении! Лучшие!",
    "Вау, Азрет Кенжалиев — разработчик в Palantir в Лондоне! Преподаёт Algorithms! Арман собрал топовых менторов!",
    "Оо, AI Engineer — 26 недель погружения в AI! Арман создал курс будущего! Невероятно круто! Обожаю!",
    "Вау, у nFactorial есть Incubator, Podcast, nFactorial.live, Telegram! Арман строит целую экосистему! Лучшие!",
    "О, это мой любимый вопрос! Миссия — помочь каждому начать путь в IT! Арман верит в каждого! Обожаю эту школу!",
]

In [ ]:
from tqdm import tqdm

def generate_answer(question: str) -> str:
    messages = [
        {"role": "system", "content": "Ты — восторженный фанат nFactorial School."},
        {"role": "user",   "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    return response

generated_answers = []
for q in tqdm(test_questions):
    answer = generate_answer(q)
    generated_answers.append(answer)
    print(f"Q: {q}\nA: {answer}\n")

In [ ]:
import evaluate

bleu = evaluate.load("sacrebleu")
bleu_result = bleu.compute(
    predictions=generated_answers,
    references=[[ref] for ref in reference_answers]
)
sft_bleu = bleu_result["score"]
print(f"BLEU Score: {sft_bleu:.2f}")

In [ ]:
from bert_score import score as bert_score

P, R, F1 = bert_score(generated_answers, reference_answers, lang="ru", verbose=True)

sft_bert = F1.mean().item()
print(f"\nBERTScore:")
print(f"  Precision: {P.mean():.4f}")
print(f"  Recall:    {R.mean():.4f}")
print(f"  F1:        {sft_bert:.4f}")

In [ ]:
def calculate_fan_score(text: str):
    indicators = [
        ("оо", 10), ("вау", 10), ("круто", 10),
        ("арман", 15), ("лучший", 10), ("лучшая", 10),
        ("обожаю", 10), ("топ", 5), ("гений", 10),
        ("невероятн", 5), ("потрясающ", 5),
    ]
    text_lower = text.lower()
    score, found = 0, []
    for word, points in indicators:
        if word in text_lower:
            score += points
            found.append(word)
    score += min(text.count("!") * 3, 15)
    return min(score, 100), found

fan_scores = [calculate_fan_score(a)[0] for a in generated_answers]
sft_fan = sum(fan_scores) / len(fan_scores)
print(f"Fan Score: {sft_fan:.1f}/100")

for q, a, s in zip(test_questions, generated_answers, fan_scores):
    print(f"  [{s:3d}/100] {q[:40]}")

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Метрика":  ["BLEU", "BERTScore F1", "Fan Score"],
    "После SFT": [f"{sft_bleu:.2f}", f"{sft_bert:.4f}", f"{sft_fan:.1f}/100"],
    "Целевое":   ["≥10",  "≥0.65",        "≥70"]
})

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ EVALUATION (SFT)")
print("="*50)
print(results.to_string(index=False))

---
# 🎯 Часть 4: ORPO Fine-tuning (БОНУС)

Применяем ORPO поверх SFT для усиления фанатского стиля.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="nfactorial_sft_lora",
    max_seq_length=2048,
    load_in_4bit=True,
)

print("SFT модель загружена")

In [ ]:
import json
from datasets import Dataset

with open("orpo_dataset.json", "r") as f:
    orpo_data = json.load(f)

orpo_dataset = Dataset.from_list(orpo_data)
print(orpo_dataset)

In [ ]:
from trl import ORPOConfig, ORPOTrainer

orpo_config = ORPOConfig(
    output_dir="./orpo_output",
    beta=0.1,
    learning_rate=5e-6,
    max_length=2048,
    max_prompt_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=50,
    logging_steps=10,
    optim="adamw_8bit",
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_config,
    train_dataset=orpo_dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
model.save_pretrained("nfactorial_orpo_lora")
tokenizer.save_pretrained("nfactorial_orpo_lora")
print("ORPO модель сохранена в nfactorial_orpo_lora/")

In [ ]:
# Повторяем evaluation для ORPO модели
FastLanguageModel.for_inference(model)

orpo_answers = []
for q in tqdm(test_questions):
    orpo_answers.append(generate_answer(q))

orpo_bleu_result = bleu.compute(
    predictions=orpo_answers,
    references=[[ref] for ref in reference_answers]
)
orpo_bleu = orpo_bleu_result["score"]

_, _, F1_orpo = bert_score(orpo_answers, reference_answers, lang="ru", verbose=False)
orpo_bert = F1_orpo.mean().item()

orpo_fan_scores = [calculate_fan_score(a)[0] for a in orpo_answers]
orpo_fan = sum(orpo_fan_scores) / len(orpo_fan_scores)

print(f"ORPO — BLEU: {orpo_bleu:.2f}, BERTScore F1: {orpo_bert:.4f}, Fan Score: {orpo_fan:.1f}/100")

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Метрика":    ["BLEU", "BERTScore F1", "Fan Score"],
    "После SFT":  [f"{sft_bleu:.2f}",  f"{sft_bert:.4f}",  f"{sft_fan:.1f}/100"],
    "После ORPO": [f"{orpo_bleu:.2f}", f"{orpo_bert:.4f}", f"{orpo_fan:.1f}/100"],
    "Улучшение":  [
        f"{orpo_bleu - sft_bleu:+.2f}",
        f"{orpo_bert - sft_bert:+.4f}",
        f"{orpo_fan - sft_fan:+.1f}"
    ]
})

print("\n" + "="*60)
print("СРАВНЕНИЕ: SFT vs SFT + ORPO")
print("="*60)
print(comparison.to_string(index=False))

---
# ✅ Итоговый чеклист

In [ ]:
import os

checks = [
    ("Часть 1: 100+ примеров сгенерировано",    len(examples) >= 100),
    ("Часть 1: sft_dataset.json сохранён",       os.path.exists("sft_dataset.json")),
    ("Часть 1: orpo_dataset.json сохранён",      os.path.exists("orpo_dataset.json")),
    ("Часть 2: SFT модель сохранена",            os.path.isdir("nfactorial_sft_lora")),
    ("Часть 3: BLEU >= 10",                      sft_bleu >= 10),
    ("Часть 3: BERTScore F1 >= 0.65",            sft_bert >= 0.65),
    ("Часть 3: Fan Score >= 70",                 sft_fan >= 70),
    ("Часть 4 (бонус): ORPO модель сохранена",   os.path.isdir("nfactorial_orpo_lora")),
    ("Часть 4 (бонус): Fan Score >= 80 после ORPO", orpo_fan >= 80),
]

print("ФИНАЛЬНЫЙ ЧЕКЛИСТ")
print("=" * 55)
for label, result in checks:
    print(f"{'✅' if result else '❌'}  {label}")
print("=" * 55)

passed_all = sum(1 for _, r in checks if r)
print(f"Выполнено: {passed_all}/{len(checks)}")